# Chinchilla — Training Compute-Optimal Large Language Models (2022)

_Papers · Transformers & LLMs_

**Given a fixed compute budget, grow model size and training tokens equally — most big models were undertrained.**

---

This notebook is a practice scaffold for the **Chinchilla — Training Compute-Optimal Large Language Models (2022)** lesson. We work through the compute relation `C = 6·N·D` and a synthetic fixed-budget loss curve one step at a time, so you can see *why* a balanced split of compute wins. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python

### Step 1 — The compute relation C = 6·N·D

The paper's rule of thumb is that training a transformer costs about `6·N·D` FLOPs, where `N` is the parameter count and `D` is the number of training tokens. Plugging in Chinchilla's own numbers (70B parameters, 1.4T tokens) gives its total training budget. This single relation is what ties model size and data together under a fixed budget.

In [ ]:
import numpy as np

# === (1) Worked example: the compute relation C = 6*N*D (paper Sec 3.3 / App. F). ===
N_chin = 70e9        # Chinchilla parameters (Table 1)
D_chin = 1.4e12      # Chinchilla training tokens (Table 1)
C = 6 * N_chin * D_chin
print("Chinchilla: N = %.1e params, D = %.1e tokens" % (N_chin, D_chin))
print("  C = 6*N*D = %.3e FLOPs" % C)        # -> 5.880e+23

### Step 2 — Spend the same budget on a bigger model

Hold the budget `C` fixed and ask: if we instead built a Gopher-sized 280B model, how many tokens could we afford? Since `D = C / (6·N)`, quadrupling the parameters quarters the tokens. The 4× ratio shows the bind Chinchilla exploits — a smaller model can read far more text for the same compute.

In [ ]:
# Same budget C, but a Gopher-sized 280B model: how many tokens can it afford?
N_gopher = 280e9
D_at_gopher = C / (6 * N_gopher)
ratio = D_chin / D_at_gopher
print("At the SAME budget, a 280B model affords D = %.3e tokens" % D_at_gopher)  # ~3.5e11
print("  ratio (Chinchilla tokens / Gopher-size tokens) = %.1fx" % ratio)
# Chinchilla: N = 7.0e+10 params, D = 1.4e+12 tokens
#   C = 6*N*D = 5.880e+23 FLOPs
# At the SAME budget, a 280B model affords D = 3.500e+11 tokens
#   ratio (Chinchilla tokens / Gopher-size tokens) = 4.0x

### Step 3 — A synthetic fixed-budget loss curve

To see the optimum, we use a **synthetic** toy loss `L = E + A·N^(-alpha) + B·D^(-beta)` with `D` forced by the fixed budget. The constants are arbitrary, chosen only to show the U-shape — they are **not** the paper's fitted values or measured loss. Scanning `N` in log-space, the minimum sits at an interior point: neither the smallest nor the largest model.

In [ ]:
# === (2) SYNTHETIC fixed-compute loss-vs-N curve (ILLUSTRATION ONLY). ===
# Toy loss = E + A*N^(-alpha) + B*D^(-beta), with D = C_fixed/(6N).
# A, B, E, alpha, beta are ARBITRARY toy constants chosen to show the U-shape.
# These are NOT the paper's fitted constants and NOT measured loss values.
C_fixed = 6 * 60e9 * 1.2e12          # a round fixed budget ~4.32e23 FLOPs
A, B, E = 90.0, 90.0, 1.2            # synthetic
alpha, beta = 0.30, 0.30             # synthetic exponents (NOT the paper's a, b)

Ns = np.logspace(np.log10(1e9), np.log10(4e12), 200)   # 1B .. 4T params
Ds = C_fixed / (6 * Ns)                                # forced by the fixed budget
L = E + A * Ns**(-alpha) + B * Ds**(-beta)

i = int(np.argmin(L))
print("\nSynthetic fixed-budget curve (illustration only):")
print("  interior optimum near N = %.2e params, D = %.2e tokens, toy loss = %.4f"
      % (Ns[i], Ds[i], L[i]))
# interior optimum near N = 2.66e+11 params, D = 2.70e+11 tokens, toy loss = 1.2671
# (Synthetic illustration of the trade-off shape, NOT the paper's measured surface.)

## Visualize the data & results

_At a FIXED compute budget (so more parameters force fewer training tokens), how does loss vary with model size — and where is the optimum?_

### Step 1 — Build the synthetic fixed-budget curve

Same synthetic toy loss as the reference cell (arbitrary constants, **not** the paper's measured surface). We sample 13 model sizes from 1B to 4T parameters in log-space; each one fixes the token count via `D = C/(6N)`, and we evaluate the toy loss `E + A·N^(-alpha) + B·D^(-beta)` at each.

In [ ]:
import numpy as np
# SYNTHETIC illustration of the fixed-compute trade-off. ARBITRARY toy constants.
# This is NOT the paper's measured loss surface or its fitted a, b constants.
C_fixed = 6 * 60e9 * 1.2e12          # fixed budget ~4.32e23 FLOPs
A, B, E = 90.0, 90.0, 1.2            # synthetic toy constants
alpha, beta = 0.30, 0.30             # synthetic toy exponents (NOT the paper's)

# 13 points in log-space from 1B to 4T parameters; D forced by the fixed budget.
xs = np.logspace(np.log10(1e9), np.log10(4e12), 13)   # N in params
Ds = C_fixed / (6 * xs)                                # D = C/(6N)
ys = E + A * xs**(-alpha) + B * Ds**(-beta)            # toy loss

### Step 2 — Print the curve and locate the interior minimum

List each `(N in billions, toy loss)` point and find where the loss bottoms out. It falls then rises along the fixed-budget line, so the minimum is interior — a balanced split of compute between model size and data. That U-shape is the whole point; the numbers are illustrative, not the paper's.

In [ ]:
for x, y in zip(xs, ys):
    print("  [%.4g, %.4f]," % (x / 1e9, y))            # x printed in billions of params
i = int(np.argmin(ys))
print("interior optimum near N = %.1fB params, toy loss = %.4f" % (xs[i] / 1e9, ys[i]))
# interior optimum near N = 252.0B params, toy loss = 1.2671
# Synthetic illustration of the trade-off shape, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The budget trade-off. You have a fixed compute budget of $C = 1.2 \times 10^{24}$ FLOPs. Using
            $C \approx 6ND$, how many training tokens $D$ can you afford for a $100$-billion-parameter model
            ($N = 1.0 \times 10^{11}$)? And for a $25$-billion-parameter model ($N = 2.5 \times 10^{10}$)? What
            does the comparison tell you?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Solve the relation for $D$: from $C \approx 6ND$, $D = C / (6N)$. — _The budget is fixed, so once $N$ is chosen, $D$ is determined &mdash; that is the whole trade-off._
- Plug in $N = 1.0 \times 10^{11}$: $D = 1.2 \times 10^{24} / (6 \times 1.0 \times 10^{11}) = 1.2 \times 10^{24} / (6.0 \times 10^{11}) = 2.0 \times 10^{12}$ tokens (2 trillion). — _Dividing the budget by the per-token cost $6N$ gives the affordable token count._
- Plug in $N = 2.5 \times 10^{10}$: the per-token cost is $6N = 1.5 \times 10^{11}$, so $D = 1.2 \times 10^{24} / (1.5 \times 10^{11}) = 8.0 \times 10^{12}$ tokens (8 trillion). — _Quartering the model size lets you afford 4&times; the tokens at the same cost &mdash; the inverse relationship between $N$ and $D$._

**Answer:** The 100B model affords $D = 2.0 \times 10^{12} = 2$ trillion tokens; the 25B model affords
                 $D = 8.0 \times 10^{12} = 8$ trillion tokens. Cutting $N$ by 4&times; buys 4&times; the data,
                 because $D = C/(6N)$ is inversely proportional to $N$ at fixed $C$. This is exactly the bind
                 Chinchilla exploits: a smaller model reads far more text for the same compute. The paper's point
                 is that the lowest-loss choice is neither extreme but a balanced interior point with
                 $N$ and $D$ scaled equally (&sect;3).

</details>

**Problem 2.** Equal-scaling consequence. The paper finds $N_{\text{opt}} \propto C^{a}$ and
            $D_{\text{opt}} \propto C^{b}$ with $a \approx b \approx 0.5$. If you quadruple your compute
            budget (multiply $C$ by 4), by what factor should the compute-optimal model size and token count each
            grow?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use $N_{\text{opt}} \propto C^{0.5}$. Multiplying $C$ by 4 multiplies $N_{\text{opt}}$ by $4^{0.5}$. — _A power law with exponent $0.5$ means the output scales as the square root of the input._
- Compute $4^{0.5} = \sqrt{4} = 2$. So $N_{\text{opt}}$ doubles. — _Square root of 4 is 2._
- By the same exponent $b \approx 0.5$, $D_{\text{opt}}$ also scales by $4^{0.5} = 2$ &mdash; it doubles too. Check against the budget: $N$ doubled $\times$ $D$ doubled $= 4\times$, matching the $4\times$ compute. — _Equal exponents mean both grow by the same factor; their product matches the budget increase, confirming $a + b \approx 1$._

**Answer:** Both double. With $a \approx b \approx 0.5$, multiplying $C$ by 4 multiplies each of
                 $N_{\text{opt}}$ and $D_{\text{opt}}$ by $4^{0.5} = 2$. This is the Abstract's rule restated:
                 "for every doubling of model size the number of training tokens should also be doubled." And the
                 two factors multiply to $2 \times 2 = 4$, consistent with the $4\times$ compute via
                 $C \approx 6ND$. Growing the model alone (and freezing data) would put the whole $4\times$ into
                 $N$ &mdash; the off-optimum mistake the paper diagnoses.

</details>

**Problem 3.** Why an interior optimum? Along a fixed-budget line ($C$ constant, so $D = C/(6N)$), explain why
            the training loss is high at both a very small $N$ and a very large $N$, forcing the minimum to
            sit in the interior. Tie this to the CODEVIZ curve below.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Consider very small $N$ (tiny model, huge $D$). The model lacks the capacity (the parameters) to represent the patterns in all that data, so it underfits and loss stays high. — _Too few parameters cap how much the model can learn no matter how much data it sees._
- Consider very large $N$ (huge model, tiny $D$). At fixed $C$, a giant model can only afford a sliver of data, so it has too little text to learn from &mdash; the undertrained regime. Loss is high again. — _$D = C/(6N)$ shrinks as $N$ grows; starve a big model of tokens and it cannot be trained well._
- Conclude: loss falls then rises along the line, so the minimum is interior &mdash; a balanced $N$ and $D$. That is the synthetic U-shape in the CODEVIZ panel. — _A function high at both ends and lower in between has its minimum in the interior; that interior point is the compute-optimal allocation._

**Answer:** At small $N$ the model is too small to use its abundant data (underfitting); at large $N$ the
                 fixed budget leaves almost no data, so the model is undertrained. Loss is high at both ends, so the
                 minimum sits in the interior &mdash; a balanced split of compute between size and data. That
                 interior minimum is precisely what "$a \approx b \approx 0.5$" pins down, and it is the bottom of
                 the synthetic U-shaped curve in the CODEVIZ panel. Reminder: that curve uses arbitrary toy
                 constants to show the shape; it is not the paper's measured loss.

</details>